In [3]:
import tensorflow as tf

In [6]:
import os
import pickle
import numpy as np

from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.models import Model

model = InceptionV3(weights='imagenet')
model = Model(model.input, model.layers[-2].output)

features = {}

image_dir = r"C:\Users\ASUS\OneDrive\Desktop\240\Flickr_Data\Images"

for img_name in os.listdir(image_dir):

    img_path = os.path.join(image_dir, img_name)

    image = load_img(img_path, target_size=(299,299))

    image = img_to_array(image)

    image = np.expand_dims(image, axis=0)

    image = preprocess_input(image)

    feature = model.predict(image, verbose=0)

    features[img_name] = feature

pickle.dump(features, open("features.pkl","wb"))

print("Features Saved")

Features Saved


In [61]:
import pandas as pd
caption_file = r"C:\Users\ASUS\OneDrive\Desktop\240\Flickr_Data\Flickr_TextData\Flickr8k.token.txt"
captions = {}

with open(r"C:\Users\ASUS\OneDrive\Desktop\240\Flickr_Data\Flickr_TextData\Flickr8k.token.txt","r") as f:

    next(f)

    for line in f:

        tokens = line.strip().split(",")

        image_id = tokens[0]

        caption = ",".join(tokens[1:])

        if image_id not in captions:
            captions[image_id] = []

        captions[image_id].append(caption)

In [60]:
import string

def clean_caption(text):

    text = text.lower()

    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    text = text.split()

    text = [word for word in text if len(word) > 1]

    return ' '.join(text)

for key in captions:
    captions[key] = [
        "startseq " +
        clean_caption(cap) +
        " endseq"
        for cap in captions[key]
    ]

In [59]:
import os

folder = r"C:\Users\ASUS\OneDrive\Desktop\240\Flickr_Data"

for item in os.listdir(folder):
    print(item)
import os

print(os.listdir(r"C:\Users\ASUS\OneDrive\Desktop\240\Flickr_Data\Flickr_TextData"))

flickr8ktextfiles
Flickr_TextData
Images
['CrowdFlowerAnnotations.txt', 'ExpertAnnotations.txt', 'Flickr8k.lemma.token.txt', 'Flickr8k.token.txt', 'Flickr_8k.devImages.txt', 'Flickr_8k.testImages.txt', 'Flickr_8k.trainImages.txt', 'readme.txt']


In [62]:
import string

def clean_caption(text):

    text = text.lower()

    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    text = text.split()

    text = [word for word in text if len(word) > 1]

    return ' '.join(text)

for key in captions:
    captions[key] = [
        "startseq " +
        clean_caption(cap) +
        " endseq"
        for cap in captions[key]
    ]

In [63]:
from tensorflow.keras.preprocessing.text import Tokenizer

all_captions = []

for key in captions:
    all_captions.extend(captions[key])

tokenizer = Tokenizer()

tokenizer.fit_on_texts(all_captions)

vocab_size = len(tokenizer.word_index)+1

In [64]:
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import add

from tensorflow.keras.models import Model

def build_model(vocab_size,max_length):

    inputs1 = Input(shape=(2048,))
    fe1 = Dropout(0.4)(inputs1)
    fe2 = Dense(256,activation='relu')(fe1)

    inputs2 = Input(shape=(max_length,))
    se1 = Embedding(
        vocab_size,
        256,
        mask_zero=True
    )(inputs2)

    se2 = Dropout(0.4)(se1)

    se3 = LSTM(256)(se2)

    decoder1 = add([fe2,se3])

    decoder2 = Dense(
        256,
        activation='relu'
    )(decoder1)

    outputs = Dense(
        vocab_size,
        activation='softmax'
    )(decoder2)

    model = Model(
        inputs=[inputs1,inputs2],
        outputs=outputs
    )

    model.compile(
        loss='categorical_crossentropy',
        optimizer='adam'
    )

    return model

In [67]:
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

def data_generator(
    captions,
    features,
    tokenizer,
    max_length,
    vocab_size
):

    while True:

        for key, desc_list in captions.items():

            feature = features[key][0]

            for desc in desc_list:

                seq = tokenizer.texts_to_sequences([desc])[0]

                for i in range(1,len(seq)):

                    in_seq = seq[:i]

                    out_seq = seq[i]

                    in_seq = pad_sequences(
                        [in_seq],
                        maxlen=max_length
                    )[0]

                    out_seq = to_categorical(
                        [out_seq],
                        num_classes=vocab_size
                    )[0]

                    yield (
                        (
                            np.array([feature]),
                            np.array([in_seq])
                        ),
                        np.array([out_seq])
                    )

In [23]:
from tensorflow.keras.layers import Input, Dense, LSTM
from tensorflow.keras.layers import Embedding, Dropout, add
from tensorflow.keras.models import Model

def build_model(vocab_size, max_length):

    inputs1 = Input(shape=(2048,))
    fe1 = Dropout(0.4)(inputs1)
    fe2 = Dense(256, activation='relu')(fe1)

    inputs2 = Input(shape=(max_length,))
    se1 = Embedding(
        vocab_size,
        256,
        mask_zero=True
    )(inputs2)

    se2 = Dropout(0.4)(se1)
    se3 = LSTM(256)(se2)

    decoder1 = add([fe2, se3])
    decoder2 = Dense(256, activation='relu')(decoder1)

    outputs = Dense(
        vocab_size,
        activation='softmax'
    )(decoder2)

    model = Model(
        inputs=[inputs1, inputs2],
        outputs=outputs
    )

    model.compile(
        loss='categorical_crossentropy',
        optimizer='adam'
    )

    return model

In [42]:
captions = {}

with open(caption_file, "r", encoding="utf-8") as f:
    for line in f:
        tokens = line.strip().split('\t')

        if len(tokens) != 2:
            continue

        image_id, caption = tokens

        # Remove #0, #1, #2...
        image_id = image_id.split('#')[0]

        if image_id not in captions:
            captions[image_id] = []

        captions[image_id].append(caption)

print("Images:", len(captions))

Images: 8092


In [32]:
model = build_model(vocab_size, max_length)
model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, 25)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_6       │ (None, 2048)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 25, 256)   │    542,208 │ input_layer_7[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 2048)      │          0 │ input_layer_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 25, 256)   │          0 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 25)        │          0 │ input_layer_7[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 256)       │    524,544 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 256)       │    525,312 │ dropout_5[0][0],  │
│                     │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 256)       │          0 │ dense_6[0][0],    │
│                     │                   │            │ lstm_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 256)       │     65,792 │ add_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 2118)      │    544,326 │ dense_7[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,202,182 (8.40 MB)

 Trainable params: 2,202,182 (8.40 MB)

 Non-trainable params: 0 (0.00 B)

In [35]:
from model import build_model

model = build_model(
    vocab_size,
    max_length
)

steps = 1000

model.fit(
    generator,
    epochs=20,
    steps_per_epoch=steps
)

model.save("caption_model.h5")

ModuleNotFoundError: No module named 'model'

In [43]:
generator = data_generator(
    captions,
    features,
    tokenizer,
    max_length,
    vocab_size
)

In [44]:
model.fit(
    generator,
    epochs=20,
    steps_per_epoch=1000
)

Epoch 1/20
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 34s 30ms/step - loss: 5.8646
Epoch 2/20
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 29s 29ms/step - loss: 5.7791
Epoch 3/20
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 29s 29ms/step - loss: 5.7494
Epoch 4/20
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 29s 29ms/step - loss: 5.3715
Epoch 5/20
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 29s 29ms/step - loss: 5.2726
Epoch 6/20
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 29s 29ms/step - loss: 5.6790
Epoch 7/20
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 29s 29ms/step - loss: 5.3821
Epoch 8/20
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 28s 28ms/step - loss: 5.3161
Epoch 9/20
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 29s 29ms/step - loss: 5.4436
Epoch 10/20
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 28s 28ms/step - loss: 5.3510
Epoch 11/20
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 34s 34ms/step - loss: 5.6663
Epoch 12/20
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 38s 38ms/step - loss: 5.3326
Epoch 13/20
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 53s 53ms/step - loss: 5.1906
Epoch 14/20
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 57s 57ms/step - loss: 5.3934
E

In [40]:
Features: 8091
Captions: 8091

In [45]:
model.save("caption_model.h5")

In [46]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

def idx_to_word(integer, tokenizer):

    for word,index in tokenizer.word_index.items():

        if index == integer:
            return word

    return None

def predict_caption(
    model,
    image_feature,
    tokenizer,
    max_length
):

    in_text = "startseq"

    for i in range(max_length):

        sequence = tokenizer.texts_to_sequences(
            [in_text]
        )[0]

        sequence = pad_sequences(
            [sequence],
            maxlen=max_length
        )

        yhat = model.predict(
            [image_feature,sequence],
            verbose=0
        )

        yhat = np.argmax(yhat)

        word = idx_to_word(
            yhat,
            tokenizer
        )

        if word is None:
            break

        in_text += " " + word

        if word == "endseq":
            break

    return in_text

In [47]:
import pickle

pickle.dump(
    tokenizer,
    open("tokenizer.pkl", "wb")
)

In [ ]:

from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

def idx_to_word(integer, tokenizer):

    for word, index in tokenizer.word_index.items():
        if index == integer:
            return word

    return None


def predict_caption(
    model,
    image_feature,
    tokenizer,
    max_length
):

    in_text = "startseq"

    for i in range(max_length):

        sequence = tokenizer.texts_to_sequences(
            [in_text]
        )[0]

        sequence = pad_sequences(
            [sequence],
            maxlen=max_length
        )

        yhat = model.predict(
            [image_feature, sequence],
            verbose=0
        )

        yhat = np.argmax(yhat)

        word = idx_to_word(
            yhat,
            tokenizer
        )

        if word is None:
            break

        in_text += " " + word

        if word == "endseq":
            break

    caption = in_text.replace(
        "startseq", ""
    ).replace(
        "endseq", ""
    )

    return caption.strip()

In [52]:
import pickle

max_length = pickle.load(
    open("max_length.pkl", "rb")
)

In [54]:
model.save("caption_model.keras")

In [55]:
print(captions[next(iter(captions))][:3])

['A child in a pink dress is climbing up a set of stairs in an entry way .', 'A girl going into a wooden building .', 'A little girl climbing into a wooden playhouse .']


In [56]:
import string

def clean_caption(text):

    text = text.lower()

    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    words = text.split()

    words = [word for word in words if len(word) > 1]

    return ' '.join(words)

for key in captions:

    captions[key] = [
        "startseq " +
        clean_caption(caption) +
        " endseq"
        for caption in captions[key]
    ]

In [57]:
first_key = next(iter(captions))
print(captions[first_key][:3])

['startseq child in pink dress is climbing up set of stairs in an entry way endseq', 'startseq girl going into wooden building endseq', 'startseq little girl climbing into wooden playhouse endseq']
